#Import Statements

This section imports necessary Python modules that the Brisca game implementation relies on. Each module provides specific functionalities used throughout the code:

1. `random`: Used for random operations such as shuffling the deck and making random choices during Monte Carlo simulations.

2. `copy`: Used to create deep copies of game states, particularly within the Monte Carlo Tree Search to avoid modifying the original state during simulations.

3. `time`: Used to measure the execution time of the Monte Carlo Tree Search algorithm, allowing it to run for a specified duration.

4. `math`: Provides mathematical functions, such as the square root and logarithm, which are used in the UCB1 formula for node selection in MCTS.

5. `os`: Used for interacting with the operating system, in this case, to clear the console screen between rounds for a cleaner user interface.

In [1]:
import random
import copy
import time
import math
import os

# Card Representation

The Card class defines the building block of the Brisca game: a single playing card. It stores essential attributes such as the card's rank and suit, its inherent value within the game's rules, and the points it contributes to a player's score. The class also provides a clear string representation for easy identification and display of card information.

In [2]:
class Card:
    """Represents a Spanish playing card used in Brisca."""

    # Card ranks with their values (0-indexed for internal use)
    RANKS = ['A', '2', '3', '4', '5', '6', '7', 'S', 'C', 'R']

    # Card suits
    SUITS = ['oros', 'copas', 'espadas', 'bastos']

    # Card values in the game
    VALUES = {
        'A': 11,  # Ace
        '3': 10,
        'R': 4,   # King
        'C': 3,   # Knight (Caballo)
        'S': 2,   # Jack (Sota)
        '2': 0,
        '4': 0,
        '5': 0,
        '6': 0,
        '7': 0
    }

    # Card point values
    POINTS = {
        'A': 11,  # Ace
        '3': 10,
        'R': 4,   # King
        'C': 3,   # Knight (Caballo)
        'S': 2,   # Jack (Sota)
        '2': 0,
        '4': 0,
        '5': 0,
        '6': 0,
        '7': 0
    }

    def __init__(self, rank, suit):
        """Initialize a card with rank and suit."""
        self.rank = rank
        self.suit = suit
        self.value = self.VALUES[rank]
        self.points = self.POINTS[rank]

    def __str__(self):
        """String representation of the card."""
        return f"{self.rank} of {self.suit}"

    def __repr__(self):
        return self.__str__()

# Deck Management

The Deck class is responsible for managing the Brisca card deck. It creates a standard 40-card Spanish deck, shuffles it randomly to ensure fair play, and provides the functionality to draw cards from the top. Additionally, it keeps track of whether the deck is empty.

In [3]:
class Deck:
    """Represents a deck of Spanish cards for Brisca."""

    def __init__(self):
        """Initialize a standard deck of 40 Spanish cards."""
        self.cards = []
        for suit in Card.SUITS:
            for rank in Card.RANKS:
                self.cards.append(Card(rank, suit))
        self.shuffle()

    def shuffle(self):
        """Shuffle the deck."""
        random.shuffle(self.cards)

    def draw(self):
        """Draw a card from the deck."""
        if not self.is_empty():
            return self.cards.pop()
        return None

    def is_empty(self):
        """Check if the deck is empty."""
        return len(self.cards) == 0

    def __len__(self):
        """Return the number of cards left in the deck."""
        return len(self.cards)

# Base Player Class

This is the base class for any player in the game, whether human or AI. It manages the player's hand, the cards they have collected, and their score. It provides actions like adding a card to the hand, playing a card (which needs to be implemented by subclasses), and collecting won cards.  The player was implemented as a class so it can be a `parent` for both an AI player and the human player. Making the implementation of players more modular.

In [4]:
class Player:
    """Represents a player in Brisca game."""

    def __init__(self, name):
        """Initialize a player with a name and empty hand."""
        self.name = name
        self.hand = []
        self.collected_cards = []
        self.score = 0

    def add_card(self, card):
        """Add a card to the player's hand."""
        if card:
            self.hand.append(card)

    def play_card(self, index):
        """Play a card from the player's hand by index."""
        if 0 <= index < len(self.hand):
            return self.hand.pop(index)
        return None

    def collect_cards(self, cards):
        """Collect cards after winning a trick."""
        self.collected_cards.extend(cards)
        self.update_score()

    def update_score(self):
        """Update the player's score based on collected cards."""
        self.score = sum(card.points for card in self.collected_cards)

    def has_cards(self):
        """Check if the player has cards in hand."""
        return len(self.hand) > 0

# Human Player Implementation

The HumanPlayer class inherits from the Player class and provides specific behavior for a human player. The crucial method here is choose_card, which prompts the human player to select a card from their hand via the console, contrary to the AI that uses Monte Carlo to find the optimal card to play.

In [5]:
class HumanPlayer(Player):
    """Human player implementation."""

    def choose_card(self, trump_suit, first_card=None):
        """Allow the human player to choose a card to play."""
        print(f"\nYour hand:")
        for i, card in enumerate(self.hand):
            print(f"{i+1}. {card}")

        while True:
            try:
                choice = int(input(f"Choose a card to play (1-{len(self.hand)}): ")) - 1
                if 0 <= choice < len(self.hand):
                    return self.play_card(choice)
                else:
                    print("Invalid card index. Try again.")
            except ValueError:
                print("Please enter a number.")

# Monte Carlo Tree Search Node

This section defines the MCTSNode class, which is a building block for the Monte Carlo Tree Search algorithm used by the AI. Each node in the search tree represents a game state. It stores information about the state, its parent, the move that led to it, its children, the number of visits, the number of wins from the AI's perspective, and the untried moves from that state. It also implements the UCB1 (Upper Confidence Bound 1) formula for selecting the next node to explore.

We chose a tree-based search structure where each node represents a specific state within a Brisca game. This allows the AI to explore potential future game states resulting from different card plays. The MCTSNode stores crucial information for this exploration.

In [6]:
class MCTSNode:
    """Node for the Monte Carlo Tree Search algorithm."""

    def __init__(self, game_state, parent=None, move=None):
        """Initialize a MCTS node with game state and parent node."""
        self.game_state = game_state
        self.parent = parent
        self.move = move  # Move that led to this state
        self.children = {}  # Map of moves to child nodes
        self.visits = 0
        self.wins = 0
        self.untried_moves = self.get_untried_moves()

    def get_untried_moves(self):
        """Get the list of untried moves from this state."""
        if self.game_state.is_terminal():
            return []

        legal_moves = self.game_state.get_legal_moves()
        return [move for move in legal_moves if move not in self.children]

    def select_child(self):
        """Select a child node using UCB1 formula."""
        # UCB1 formula: wi/ni + C * sqrt(ln(N)/ni)
        c = 1.414  # Exploration parameter

        best_score = -float('inf')
        best_child = None

        for child in self.children.values():
            # Avoid division by zero
            if child.visits == 0:
                score = float('inf')
            else:
                # UCB1 calculation
                exploit = child.wins / child.visits
                explore = math.sqrt(math.log(self.visits) / child.visits)
                score = exploit + c * explore

            if score > best_score:
                best_score = score
                best_child = child

        return best_child

    def add_child(self, move, game_state):
        """Add a child node with the given move and state."""
        child = MCTSNode(game_state, self, move)
        self.children[move] = child
        return child

    def update(self, result):
        """Update node statistics with simulation result."""
        self.visits += 1
        self.wins += result

    def is_fully_expanded(self):
        """Check if all possible moves from this state have been tried."""
        return len(self.untried_moves) == 0

    def is_terminal(self):
        """Check if this node represents a terminal game state."""
        return self.game_state.is_terminal()

# AI Player with Monte Carlo Search

The AIPlayer class implements intelligent behavior in Brisca by employing the Monte Carlo Tree Search (MCTS) algorithm. We opted for MCTS due to its effectiveness in navigating complex game states with uncertain outcomes. This simulation-based approach allows the AI to learn and adapt dynamically, contrasting with rule-based systems that follow a fixed set of instructions.

In [7]:
class AIPlayer(Player):
    """AI player using Monte Carlo Tree Search."""

    def __init__(self, name, simulation_time=1.0):
        """Initialize AI player with name and simulation time."""
        super().__init__(name)
        self.simulation_time = simulation_time  # Time in seconds for MCTS

    def choose_card(self, trump_suit, first_card=None):
        """Choose a card to play using Monte Carlo Tree Search."""
        if len(self.hand) == 1:
            # If only one card in hand, just play it
            return self.play_card(0)

        # Create a game state from current information
        game_state = BriscaGameState(
            hand=self.hand.copy(),
            trump_suit=trump_suit,
            first_card=first_card
        )

        # Run Monte Carlo Tree Search
        best_move = self.monte_carlo_tree_search(game_state)

        # Find the chosen card in player's hand
        for i, card in enumerate(self.hand):
            if card == best_move:
                return self.play_card(i)

        # Fallback to first card if something goes wrong
        return self.play_card(0)

    def monte_carlo_tree_search(self, game_state):
        """Run MCTS algorithm to find the best move."""
        root = MCTSNode(game_state)
        end_time = time.time() + self.simulation_time

        # Run simulations until time limit
        while time.time() < end_time:
            # Selection
            node = self.select(root)

            # Expansion
            if not node.is_terminal() and not node.is_fully_expanded():
                node = self.expand(node)

            # Simulation
            result = self.simulate(node.game_state)

            # Backpropagation
            self.backpropagate(node, result)

        # Return the move with the most visits
        return self.best_move(root)

    def select(self, node):
        """Select a node to explore further."""
        while not node.is_terminal() and node.is_fully_expanded():
            node = node.select_child()
        return node

    def expand(self, node):
        """Expand the node by adding a child node."""
        move = random.choice(node.untried_moves)
        new_state = node.game_state.apply_move(move)
        return node.add_child(move, new_state)

    def simulate(self, game_state):
        """Simulate a random playout from the current state."""
        state = copy.deepcopy(game_state)

        while not state.is_terminal():
            legal_moves = state.get_legal_moves()
            move = random.choice(legal_moves)
            state = state.apply_move(move)

        # Return 1 for win, 0 for loss from AI perspective
        return state.get_result()

    def backpropagate(self, node, result):
        """Backpropagate the simulation result up the tree."""
        while node is not None:
            node.update(result)
            node = node.parent

    def best_move(self, root):
        """Select the best move based on visit count."""
        best_visits = -float('inf')
        best_move = None

        for move, child in root.children.items():
            if child.visits > best_visits:
                best_visits = child.visits
                best_move = move

        return best_move

# Brisca Game State for MCTS

The AIPlayer class utilizes the Monte Carlo Tree Search (MCTS) algorithm to make intelligent decisions in Brisca. We chose MCTS as our primary design because it excels in games with complex state spaces and uncertain outcomes. Unlike traditional rule-based AIs, MCTS learns through simulating numerous potential game scenarios, allowing it to adapt to various situations and discover effective strategies without explicit pre-programming of every possible rule.

The AI's decision-making process, initiated by the choose_card method, involves building and traversing a search tree of game states. Through the iterative phases of selection, expansion, simulation, and backpropagation, MCTS evaluates the potential of different moves. By simulating many random playouts and analyzing the win rates associated with each action, the AI implicitly evaluates its options and ultimately selects the move that has demonstrated the most promise during its internal simulations.

In [8]:
class BriscaGameState:
    """Represents a state of the Brisca game for MCTS."""

    def __init__(self, hand, trump_suit, first_card=None):
        """Initialize game state with player's hand and current game info."""
        self.hand = hand.copy()  # AI player's hand
        self.trump_suit = trump_suit  # Trump suit for this game
        self.first_card = first_card  # Card played by the opponent (if any)

    def get_legal_moves(self):
        """Get all legal moves (cards) that can be played."""
        # In Brisca, any card can be played
        return self.hand

    def apply_move(self, move):
        """Apply a move (play a card) and return the resulting state."""
        new_hand = self.hand.copy()
        new_hand.remove(move)

        # If we're playing first, first_card becomes our move
        if self.first_card is None:
            return BriscaGameState(new_hand, self.trump_suit, move)
        else:
            # If we're playing second, this is a terminal state
            # The first_card remains as the opponent's card
            return BriscaGameState(new_hand, self.trump_suit, self.first_card)

    def is_terminal(self):
        """Check if this is a terminal state."""
        # If we're playing second, this is a terminal state after our move
        return self.first_card is not None and len(self.hand) < len(self.hand) + 1

    def get_result(self):
        """Evaluate the terminal state from AI's perspective."""
        if not self.is_terminal() or self.first_card is None:
            return 0  # Not a terminal state or we played first

        # If we're here, we played second and need to determine who won the trick
        # Assume the last card in hand was played
        our_card = self.hand[0] if len(self.hand) > 0 else None

        if our_card:
            winner = self.determine_winner(self.first_card, our_card, self.trump_suit)
            # Return 1 if AI wins, 0 if opponent wins
            return 1 if winner == our_card else 0
        return 0

    @staticmethod
    def determine_winner(card1, card2, trump_suit):
        """Determine the winning card in a trick."""
        # If second card is trump and first is not, second card wins
        if card2.suit == trump_suit and card1.suit != trump_suit:
            return card2

        # If first card is trump and second is not, first card wins
        if card1.suit == trump_suit and card2.suit != trump_suit:
            return card1

        # If both cards have the same suit, the higher value wins
        if card1.suit == card2.suit:
            return card1 if card1.value > card2.value else card2

        # If different suits and no trump, first card wins
        return card1

# Brisca Game Logic

The BriscaGame class serves as the central controller for a game of Brisca. Its design focuses on managing all aspects of the game flow, from initializing the deck and players to determining the final winner. This class maintains the game state, including the players involved, the current deck of cards, the crucial trump card and suit, and whose turn it is.

The BriscaGame encompasses the core logic of playing Brisca. It handles the progression of rounds (tricks), determines the winner of each trick based on card values and the trump suit, and manages the distribution of new cards to players. Furthermore, it includes the necessary checks to identify when the game has concluded and provides the functionality to calculate and display the final scores, ultimately declaring the winner of the match.

In [9]:
class BriscaGame:
    """Main class representing a game of Brisca."""

    def __init__(self, player1, player2):
        """Initialize a new game with two players."""
        self.players = [player1, player2]
        self.deck = Deck()
        self.current_player_idx = 0  # Player 1 starts
        self.trump_card = None
        self.trump_suit = None
        self.initialize_game()

    def initialize_game(self):
        """Initialize the game by dealing cards and setting trump."""
        # Draw trump card
        self.trump_card = self.deck.draw()
        self.trump_suit = self.trump_card.suit

        # Deal three cards to each player
        for _ in range(3):
            for player in self.players:
                player.add_card(self.deck.draw())

    def play_round(self):
        """Play a round of Brisca."""
        trick_cards = []
        first_player_idx = self.current_player_idx

        # First player plays a card
        first_player = self.players[first_player_idx]
        first_card = first_player.choose_card(self.trump_suit)
        print(f"\n{first_player.name} plays: {first_card}")
        trick_cards.append(first_card)

        # Second player plays a card
        second_player_idx = (first_player_idx + 1) % 2
        second_player = self.players[second_player_idx]
        second_card = second_player.choose_card(self.trump_suit, first_card)
        print(f"{second_player.name} plays: {second_card}")
        trick_cards.append(second_card)

        # Determine winner of the trick
        winner_idx = self.determine_trick_winner(first_card, second_card, first_player_idx)
        winner = self.players[winner_idx]
        winner.collect_cards(trick_cards)
        print(f"{winner.name} wins the trick!")

        # Winner draws first, then loser
        self.deal_new_cards(winner_idx)

        # Winner plays next
        self.current_player_idx = winner_idx

        return not self.is_game_over()

    def determine_trick_winner(self, card1, card2, first_player_idx):
        """Determine which player wins the trick."""
        # Trump suit logic
        if card1.suit == self.trump_suit and card2.suit != self.trump_suit:
            return first_player_idx
        if card2.suit == self.trump_suit and card1.suit != self.trump_suit:
            return (first_player_idx + 1) % 2

        # Same suit logic - higher value wins
        if card1.suit == card2.suit:
            if card1.value > card2.value:
                return first_player_idx
            else:
                return (first_player_idx + 1) % 2

        # Different suits and no trump involved - first card wins
        return first_player_idx

    def deal_new_cards(self, winner_idx):
        """Deal new cards to players after a trick."""
        # Winner draws first, then loser
        for i in range(2):
            player_idx = (winner_idx + i) % 2
            if len(self.players[player_idx].hand) < 3 and not self.deck.is_empty():
                # If trump card is the last card, give it to the player
                if len(self.deck) == 1 and self.trump_card:
                    self.players[player_idx].add_card(self.trump_card)
                    self.trump_card = None
                else:
                    self.players[player_idx].add_card(self.deck.draw())

    def is_game_over(self):
        """Check if the game is over."""
        return self.deck.is_empty() and not self.trump_card and all(
            len(player.hand) == 0 for player in self.players
        )

    def get_winner(self):
        """Get the winner of the game."""
        if self.players[0].score > self.players[1].score:
            return self.players[0]
        elif self.players[1].score > self.players[0].score:
            return self.players[1]
        else:
            return None  # Draw

    def display_game_status(self):
        """Display the current status of the game."""
        print("\n=== Game Status ===")
        print(f"Trump card: {self.trump_card}" if self.trump_card else "Trump card: [Used]")
        print(f"Trump suit: {self.trump_suit}")
        print(f"Cards left in deck: {len(self.deck)}")
        print(f"Current player: {self.players[self.current_player_idx].name}")

        # Display scores
        for player in self.players:
            print(f"{player.name}'s score: {player.score}")

    def play_game(self):
        """Play a full game of Brisca."""
        print("=== Starting a new game of Brisca ===")
        print(f"Trump suit: {self.trump_suit} (from {self.trump_card})")

        # Continue playing rounds until the game is over
        round_num = 1
        while True:
            print(f"\n=== Round {round_num} ===")
            self.display_game_status()

            if not self.play_round():
                break

            round_num += 1

            input("Press Enter to continue...")
            os.system('cls' if os.name == 'nt' else 'clear')  # Clear console for next round

        # Game over, display final results
        print("\n=== Game Over ===")
        print(f"Final scores:")
        for player in self.players:
            print(f"{player.name}: {player.score}")

        winner = self.get_winner()
        if winner:
            print(f"The winner is {winner.name}!")
        else:
            print("The game ended in a draw!")

# Main Game Execution

This final section contains the main function, which is the entry point of the script. It initializes the game by prompting the user for their name, creating instances of HumanPlayer and AIPlayer, and then creating and starting the BriscaGame. It also includes a closing message after the game ends.

In [ ]:
def main():
    """Main function to run the game."""
    print("Welcome to Brisca!")
    print("Would you like to play against the AI?")

    # Create players
    player_name = input("Enter your name: ")
    human_player = HumanPlayer(player_name)
    ai_player = AIPlayer("Monte Carlo AI")

    # Create and play the game
    game = BriscaGame(human_player, ai_player)
    game.play_game()

    # AI VS AI
    # ai_player_1 = AIPlayer("Optimus")
    # ai_player_2 = AIPlayer("Megatron")

    # Create and play the game
    # game = BriscaGame(ai_player_1, ai_player_2)
    # game.play_game()

    print("Thanks for playing!")


if __name__ == "__main__":
    main()

Welcome to Brisca!
Would you like to play against the AI?
=== Starting a new game of Brisca ===
Trump suit: oros (from A of oros)

=== Round 1 ===

=== Game Status ===
Trump card: A of oros
Trump suit: oros
Cards left in deck: 33
Current player: Manuel
Manuel's score: 0
Monte Carlo AI's score: 0

Your hand:
1. S of espadas
2. S of copas
3. 2 of copas

Manuel plays: S of espadas
Monte Carlo AI plays: A of espadas
Monte Carlo AI wins the trick!

=== Round 2 ===

=== Game Status ===
Trump card: A of oros
Trump suit: oros
Cards left in deck: 31
Current player: Monte Carlo AI
Manuel's score: 0
Monte Carlo AI's score: 13

Monte Carlo AI plays: S of bastos

Your hand:
1. S of copas
2. 2 of copas
3. A of bastos
Manuel plays: A of bastos
Manuel wins the trick!

=== Round 3 ===

=== Game Status ===
Trump card: A of oros
Trump suit: oros
Cards left in deck: 29
Current player: Manuel
Manuel's score: 13
Monte Carlo AI's score: 13

Your hand:
1. S of copas
2. 2 of copas
3. C of copas


# Assignment Report

  This project implements a Python-based simulation of the traditional Spanish card game, Brisca, featuring an Artificial Intelligence (AI) opponent powered by a Monte Carlo Tree Search (MCTS) algorithm. The primary purpose of this project is to develop a robust decision-making framework for a partially observable environment and compare its efficiency against other adversarial search techniques. The AI evaluates potential future game states through iterative simulations, balancing risk and reward to select optimal card plays without the need for pre-trained data.
  
  Brisca is a trick-taking card game played with a standard 40-card Spanish deck. Developing an AI for this game presents a unique challenge because Brisca is a partially observable environment; players do not have perfect knowledge of their opponent's hand or the unrevealed cards in the deck. The scope of this project includes building the core object-oriented game logic, creating human and AI player mechanics, and implementing a Monte Carlo Tree Search to guide the AI's decision-making process.
  
  The technical foundation of this project relies on two main components: the rules of Brisca and the Monte Carlo Tree Search algorithm.
  
  Brisca Mechanics: The game uses a 40-card deck with four suits (oros, copas, espadas, bastos) and specific point values assigned to card ranks. Aces are worth 11 points, Threes are worth 10, Kings are 4, Knights are 3, and Jacks are 2, while all other cards have a value of 0. Tricks are won based on a designated "trump" suit or by playing the highest value card of the leading suit.
  
  Adversarial Search Options: MCTS was selected over other search methods (such as stochastic games, partially observable games, averaging over clairvoyance, or deep neural networks) because of its specific advantages. Stochastic games and partially observable game searches require keeping track of billions of potential state combinations, resulting in absurdly high complexity. Deep neural networks require extensive training data. Conversely, MCTS scales linearly, requires no prior training, and excels in adaptable, partially observable environments.
  
  Monte Carlo Tree Search (MCTS): MCTS is a simulation-based technique that involves creating probability distributions of possible outcomes to quantitatively assess risk. It explores potential future game states resulting from different card plays using the UCB1 (Upper Confidence Bound) formula to balance the exploitation of known winning moves with the exploration of untried ones: $w_i/n_i + C \cdot \sqrt{\ln(N)/n_i}$.
  
  The project was developed in Python, utilizing standard libraries such as random for shuffling and simulation choices, copy to duplicate game states without modifying the original, time to manage simulation duration, and math to compute the UCB1 formula.Approach and ImplementationGame State Management: The game was built using an object-oriented approach, starting with a Card class to hold rank, suit, and point values, and a Deck class to manage the 40-card collection. A central BriscaGame class handles dealing cards, determining the trump suit, evaluating trick winners, and tracking overall game progression.Player Modularity: A base Player class was established to manage hands and scorekeeping. This was extended into a HumanPlayer class that takes console input, and an AIPlayer class that utilizes the MCTS algorithm to make automated decisions.MCTS Architecture: The AI constructs a search tree using the MCTSNode class, which stores game states, parent/child node relationships, visit counts, and win statistics. During its turn, the AI runs a continuous loop for a designated simulation time (defaulting to 1.0 second), executing the four phases of MCTS: selection (traversing to an optimal node), expansion (adding a new game state child node), simulation (playing out a random game to a terminal state), and backpropagation (updating the win/visit statistics back up the tree). Once the time limit is reached, the AI selects the move associated with the highest visit count.







# References
[1] Monte Carlo Simulation Explained, Vose Software, 2023. [Online]. Available: https://www.vosesoftware.com/Articles/Monte-Carlo-simulation-explained.php. [Accessed: May 8, 2026].

[2] Stochastic Games in Artificial Intelligence, GeeksforGeeks, (n.d.). [Online]. Available: https://www.geeksforgeeks.org/stochastic-games-in-artificial-intelligence/. [Accessed: May 8, 2026].

[3] Partially Observable Games, QuantumComputers.in, (n.d.). [Online]. Available: https://www.quantumcomputers.co.in/viewtopic.php?t=173#:~:text=Partially%20Observable%20Games%2C%20often%20referred,the%20underlying%20state%20of%20the. [Accessed: May 8, 2026].

[4] S. Russell and P. Norvig, Artificial Intelligence: A Modern Approach, 3rd ed. Prentice Hall, 2010, ch. 5. [Online]. Available: http://cs.gettysburg.edu/~tneller/fys187-4/aima3e-ch5.pdf. [Accessed: May 8, 2026].

[5] Deep Neural Network, ScienceDirect, (n.d.). [Online]. Available: https://www.sciencedirect.com/topics/computer-science/deep-neural-network. [Accessed: May 8, 2026].